# RSAM-Seg Notebook 运行模板

这个 notebook 的目标是：
- 在 notebook 里生成一份本地可用的配置文件
- 直接通过 `!python train.py ...` 启动训练
- 直接通过 `!python test.py ...` 做评估

运行前请确认：
1. 你在仓库根目录打开 notebook
2. 你有 CUDA GPU
3. 你已经准备好数据集目录和 `pretrained/sam_vit_b_01ec64.pth`


In [ ]:
from pathlib import Path
import sys
import yaml

REPO_ROOT = Path.cwd()
TEMPLATE_CONFIG = REPO_ROOT / 'configs' / 'cod-sam-vit-b.notebook.yaml'
RUNTIME_CONFIG = REPO_ROOT / 'configs' / 'cod-sam-vit-b.notebook.runtime.yaml'
RUN_NAME_PREFIX = 'notebook_demo'
SEEDS = [42, 123, 456]
CUDA_VISIBLE_DEVICES = '0,1'
NPROC_PER_NODE = 2
EVAL_TYPE = 'seg'
PYTHON_BIN = sys.executable

TRAIN_IMAGE_DIR = './pv08_prepared/train/images'
TRAIN_MASK_DIR = './pv08_prepared/train/masks'
VAL_IMAGE_DIR = './pv08_prepared/val/images'
VAL_MASK_DIR = './pv08_prepared/val/masks'
SAM_CHECKPOINT = str(REPO_ROOT / 'pretrained' / 'sam_vit_b_01ec64.pth')
EPOCH_MAX = 120

print('Repo root:', REPO_ROOT)
print('Template config:', TEMPLATE_CONFIG)
print('Runtime config:', RUNTIME_CONFIG)
print('Run name prefix:', RUN_NAME_PREFIX)
print('Seeds:', SEEDS)
print('CUDA_VISIBLE_DEVICES:', CUDA_VISIBLE_DEVICES)
print('NPROC_PER_NODE:', NPROC_PER_NODE)
print('Eval type:', EVAL_TYPE)
print('Python bin:', PYTHON_BIN)


## 你通常只需要改上一个代码块里的这些变量

- `TRAIN_IMAGE_DIR` / `TRAIN_MASK_DIR`
- `VAL_IMAGE_DIR` / `VAL_MASK_DIR`
- `SAM_CHECKPOINT`
- `EPOCH_MAX`
- `RUN_NAME_PREFIX`
- `SEEDS`
- `CUDA_VISIBLE_DEVICES`
- `NPROC_PER_NODE`
- `EVAL_TYPE`

如果你只是想先验证链路是否跑通，可以先把 `EPOCH_MAX = 1`。


In [ ]:
with open(TEMPLATE_CONFIG, 'r', encoding='utf-8') as f:
    config = yaml.load(f, Loader=yaml.FullLoader)

config['train_dataset']['dataset']['args']['root_path_1'] = TRAIN_IMAGE_DIR
config['train_dataset']['dataset']['args']['root_path_2'] = TRAIN_MASK_DIR
config['val_dataset']['dataset']['args']['root_path_1'] = VAL_IMAGE_DIR
config['val_dataset']['dataset']['args']['root_path_2'] = VAL_MASK_DIR
config['test_dataset']['dataset']['args']['root_path_1'] = VAL_IMAGE_DIR
config['test_dataset']['dataset']['args']['root_path_2'] = VAL_MASK_DIR
config['sam_checkpoint'] = SAM_CHECKPOINT
config['epoch_max'] = EPOCH_MAX
config['eval_type'] = EVAL_TYPE

with open(RUNTIME_CONFIG, 'w', encoding='utf-8') as f:
    yaml.dump(config, f, sort_keys=False, allow_unicode=True)

print('已生成配置文件:', RUNTIME_CONFIG)


In [ ]:
paths_to_check = [
    TEMPLATE_CONFIG,
    Path(TRAIN_IMAGE_DIR),
    Path(TRAIN_MASK_DIR),
    Path(VAL_IMAGE_DIR),
    Path(VAL_MASK_DIR),
    Path(SAM_CHECKPOINT),
]

for path in paths_to_check:
    print(f'{path} -> {path.exists()}')


## 安装依赖

首次运行时，先取消下一格里的注释执行。


In [ ]:
# !pip install -r requirements.txt


## 多随机种子 + 多卡训练 + 评估

下面会按 `SEEDS` 里的每个种子依次训练和评估，并记录结果。训练使用 `CUDA_VISIBLE_DEVICES` + `NPROC_PER_NODE` 多卡 DDP。


In [ ]:
import os
import re
import subprocess

EVAL_METRIC_NAMES = {
    'cod': ('sm', 'em', 'wfm', 'mae'),
    'f1': ('f1', 'auc', 'none', 'none'),
    'fmeasure': ('f_mea', 'mae', 'none', 'none'),
    'ber': ('shadow', 'non_shadow', 'ber', 'none'),
    'seg': ('precision', 'recall', 'f1', 'iou'),
}

eval_type = config.get('eval_type', 'seg')
metric_names = EVAL_METRIC_NAMES.get(eval_type, ('metric1', 'metric2', 'metric3', 'metric4'))

RUNNER_PATH = REPO_ROOT / '_notebook_seed_runner.py'
RUNNER_PATH.write_text("""
import os
import sys
import random

seed = int(os.environ.get('RSAM_SEED', '0'))
run_name = os.environ.get('RSAM_RUN_NAME', 'run')
config_path = os.environ.get('RSAM_CONFIG', 'configs/cod-sam-vit-l.notebook.runtime.yaml')

os.environ['PYTHONHASHSEED'] = str(seed)
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':16:8')

import numpy as np
import torch
import runpy

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

sys.argv = ['train.py', '--config', config_path, '--name', run_name]
runpy.run_path('train.py', run_name='__main__')
""", encoding='utf-8')

def run_train(seed, run_name):
    print(f'==> Train seed={seed} run_name={run_name}')
    env = os.environ.copy()
    env['CUDA_VISIBLE_DEVICES'] = CUDA_VISIBLE_DEVICES
    env['RSAM_SEED'] = str(seed)
    env['RSAM_RUN_NAME'] = run_name
    env['RSAM_CONFIG'] = str(RUNTIME_CONFIG)
    cmd = [
        PYTHON_BIN, '-m', 'torch.distributed.run',
        '--standalone',
        '--nproc_per_node', str(NPROC_PER_NODE),
        str(RUNNER_PATH),
    ]
    subprocess.run(cmd, check=True, env=env)


def run_eval(model_path):
    cmd = [PYTHON_BIN, 'test.py', '--config', str(RUNTIME_CONFIG), '--model', str(model_path)]
    completed = subprocess.run(
        cmd,
        check=True,
        text=True,
        capture_output=True,
    )
    output = (completed.stdout or '') + '\n' + (completed.stderr or '')
    metrics = {}
    for key in ('metric1', 'metric2', 'metric3', 'metric4'):
        match = re.findall(rf'{key}:\\s*([0-9.]+)', output)
        if match:
            metrics[key] = float(match[-1])
    if len(metrics) < 4:
        raise RuntimeError('Failed to parse metrics from test output.')
    return metrics


results = []
for seed in SEEDS:
    run_name = f'{RUN_NAME_PREFIX}_seed{seed}'
    run_train(seed, run_name)

    best_path = REPO_ROOT / 'save' / run_name / 'model_epoch_best.pth'
    last_path = REPO_ROOT / 'save' / run_name / 'model_epoch_last.pth'
    model_path = best_path if best_path.exists() else last_path
    if not model_path.exists():
        raise FileNotFoundError(f'No checkpoint found for {run_name}: {best_path} / {last_path}')

    raw = run_eval(str(model_path))
    named = {}
    for i, name in enumerate(metric_names, start=1):
        if name == 'none':
            continue
        named[name] = raw[f'metric{i}']

    results.append({
        'seed': seed,
        'run_name': run_name,
        'model_path': str(model_path),
        **named,
    })

print('All runs finished.')


## 汇总结果

In [ ]:
import math

if not results:
    raise RuntimeError('No results collected. Please run the training cell above first.')

metric_keys = [k for k in metric_names if k != 'none']
headers = ['seed', 'run_name'] + metric_keys
rows = []
for item in results:
    row = [item['seed'], item['run_name']] + [item.get(k) for k in metric_keys]
    rows.append(row)

col_widths = [max(len(str(h)), max(len(str(r[i])) for r in rows)) for i, h in enumerate(headers)]
line = ' | '.join(str(h).ljust(col_widths[i]) for i, h in enumerate(headers))
sep = '-+-'.join('-' * w for w in col_widths)
print(line)
print(sep)
for r in rows:
    print(' | '.join(str(r[i]).ljust(col_widths[i]) for i in range(len(headers))))

print('\nMean / Std over seeds')
for k in metric_keys:
    vals = [item[k] for item in results]
    mean = sum(vals) / len(vals)
    var = sum((v - mean) ** 2 for v in vals) / len(vals)
    std = math.sqrt(var)
    print(f'{k}: mean={mean:.4f}, std={std:.4f}')


In [ ]:
results
